# Consumer Complaint Classification (NLP, Deep Learning)

End-to-end pipeline that classifies customer complaint narratives into product categories. Three sequence models built from scratch (SimpleRNN, LSTM, GRU) are compared against a fine-tuned HuggingFace DistilBERT transformer, and the best-performing model is deployed with Gradio.

## Config

In [ ]:
import os
import re
import json
import pickle
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
CONFIG = {
    "data_path": "complaints_processed.csv",
    "random_state": 42,
    "sample_size": 30_000,      # cap for training time; full clean set is ~124K rows
    "test_size": 0.2,
    "vocab_size": 20_000,
    "embedding_dim": 128,
    "rnn_units": 64,
    "rnn_epochs": 4,
    "batch_size": 128,
    "transformer_name": "distilbert-base-uncased",
    "transformer_train_size": 4_000,   # subset for fine-tuning speed
    "transformer_epochs": 2,
    "transformer_lr": 3e-5,
    "save_dir": "best_model",
}


In [ ]:
sns.set_theme(style="whitegrid")
plt.rcParams["font.size"] = 11
np.random.seed(CONFIG["random_state"])


## Stage 1 — Data Exploration

In [ ]:
df = pd.read_csv(CONFIG["data_path"])
df.head()


In [ ]:
# drop the leftover unnamed index column if it's there
df = df.drop(columns=[c for c in df.columns if "unnamed" in c.lower()])
df.shape


In [ ]:
df.isna().sum()


In [ ]:
df.duplicated(subset=["narrative"]).sum()


In [ ]:
df = df.dropna(subset=["narrative"]).drop_duplicates(subset=["narrative"]).reset_index(drop=True)
df.shape


In [ ]:
class_counts = df["product"].value_counts()
class_counts


In [ ]:
plt.figure(figsize=(9, 4))
ax = sns.barplot(x=class_counts.values, y=class_counts.index, palette="viridis")
ax.set_title("Complaint Category Distribution")
ax.set_xlabel("Count")
ax.set_ylabel("")
for p in ax.patches:
    w = p.get_width()
    ax.annotate(f"{int(w):,}", (w + 500, p.get_y() + p.get_height() / 2), va="center")
plt.tight_layout()
plt.show()


## Stage 2 — Text Preprocessing

Lowercase, strip non-letters, remove stop words, lemmatize. This is used for the RNN models. The transformer tokenizes raw text directly, since subword tokenizers handle casing and punctuation on their own.

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)


In [ ]:
STOP_WORDS = set(stopwords.words("english"))
LEMMATIZER = WordNetLemmatizer()

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"[^a-z\s]", "", text)
    tokens = [LEMMATIZER.lemmatize(w) for w in text.split() if w not in STOP_WORDS and len(w) > 1]
    return " ".join(tokens)


In [ ]:
df["clean_narrative"] = df["narrative"].apply(clean_text)
df[["narrative", "clean_narrative"]].head(3)


## Stage 3 — Class Imbalance and Train/Test Split

One stratified split produces a shared set of row indices, reused for both the cleaned text (RNNs) and the raw text (transformer), so `y_test` stays identical across all four models.

In [ ]:
print(f"Imbalance ratio (majority/minority): {class_counts.max() / class_counts.min():.2f}x")


In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["product"])
num_classes = len(label_encoder.classes_)
class_names = label_encoder.classes_
num_classes


In [ ]:
from sklearn.model_selection import train_test_split

sample_n = min(CONFIG["sample_size"], len(df))
df_sample, _ = train_test_split(
    df, train_size=sample_n, random_state=CONFIG["random_state"], stratify=df["label"]
)
df_sample = df_sample.reset_index(drop=True)
df_sample.shape


In [ ]:
# one split, reused for both the RNN (clean) text and the transformer (raw) text
train_idx, test_idx = train_test_split(
    df_sample.index,
    test_size=CONFIG["test_size"],
    random_state=CONFIG["random_state"],
    stratify=df_sample["label"],
)
train_df = df_sample.loc[train_idx].reset_index(drop=True)
test_df = df_sample.loc[test_idx].reset_index(drop=True)
y_train, y_test = train_df["label"].values, test_df["label"].values
len(train_df), len(test_df)


In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weights_dict = dict(zip(
    np.unique(y_train),
    compute_class_weight(class_weight="balanced", classes=np.unique(y_train), y=y_train),
))
{class_names[k]: round(v, 3) for k, v in class_weights_dict.items()}


## Stage 4 — Tokenization and Sequences

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer(num_words=CONFIG["vocab_size"], oov_token="<OOV>")
tokenizer.fit_on_texts(train_df["clean_narrative"])


In [ ]:
train_seq = tokenizer.texts_to_sequences(train_df["clean_narrative"])
test_seq = tokenizer.texts_to_sequences(test_df["clean_narrative"])


In [ ]:
# pick MAX_LEN from the 95th percentile so a few long outliers don't blow up padding
p95 = int(np.percentile([len(s) for s in train_seq], 95))
MAX_LEN = max(50, min(p95, 150))
MAX_LEN


In [ ]:
X_train_pad = pad_sequences(train_seq, maxlen=MAX_LEN, padding="post", truncating="post")
X_test_pad = pad_sequences(test_seq, maxlen=MAX_LEN, padding="post", truncating="post")
X_train_pad.shape, X_test_pad.shape


## Stage 5 — Embedding Layer

One embedding configuration, reused by all three RNN variants below, keeps the comparison focused on the recurrent cell rather than embedding differences.

In [ ]:
print(f"vocab_size={CONFIG['vocab_size']}, embedding_dim={CONFIG['embedding_dim']}, "
      f"input_length={MAX_LEN}, num_classes={num_classes}")


## Stage 6 — Model Development

### SimpleRNN, LSTM, GRU

Same builder function for all three, so the only difference between them is the recurrent layer itself.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, GRU, Dense
from tensorflow.keras.callbacks import EarlyStopping

def build_rnn_model(layer_cls):
    return Sequential([
        Embedding(input_dim=CONFIG["vocab_size"], output_dim=CONFIG["embedding_dim"]),
        layer_cls(CONFIG["rnn_units"], dropout=0.2),
        Dense(num_classes, activation="softmax"),
    ])


In [ ]:
RNN_LAYERS = {"SimpleRNN": SimpleRNN, "LSTM": LSTM, "GRU": GRU}
rnn_models = {}


In [ ]:
for name, layer_cls in RNN_LAYERS.items():
    print(f"--- Training {name} ---")
    model = build_rnn_model(layer_cls)
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    model.fit(
        X_train_pad, y_train,
        validation_split=0.1,
        epochs=CONFIG["rnn_epochs"],
        batch_size=CONFIG["batch_size"],
        class_weight=class_weights_dict,
        callbacks=[EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)],
        verbose=1,
    )
    rnn_models[name] = model


### Fine-tuned DistilBERT

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

hf_tokenizer = AutoTokenizer.from_pretrained(CONFIG["transformer_name"])
hf_model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG["transformer_name"], num_labels=num_classes
)


In [ ]:
# fine-tune on a subset of the training narratives for speed
trans_train_n = min(CONFIG["transformer_train_size"], len(train_df))
train_narr = train_df["narrative"].values[:trans_train_n]
train_labels = y_train[:trans_train_n]

train_enc = hf_tokenizer(list(train_narr), max_length=128, padding=True, truncation=True, return_tensors="pt")
test_enc = hf_tokenizer(list(test_df["narrative"].values), max_length=128, padding=True, truncation=True, return_tensors="pt")


In [ ]:
train_loader = DataLoader(
    TensorDataset(train_enc["input_ids"], train_enc["attention_mask"], torch.tensor(train_labels)),
    batch_size=32, shuffle=True,
)
test_loader = DataLoader(
    TensorDataset(test_enc["input_ids"], test_enc["attention_mask"]),
    batch_size=64, shuffle=False,
)


In [ ]:
# same balanced class weights used for the RNNs, applied here directly to the loss
weight_tensor = torch.tensor(
    [class_weights_dict[c] for c in range(num_classes)], dtype=torch.float32
)
loss_fn = nn.CrossEntropyLoss(weight=weight_tensor)
optimizer = torch.optim.AdamW(hf_model.parameters(), lr=CONFIG["transformer_lr"])


In [ ]:
hf_model.train()
for epoch in range(CONFIG["transformer_epochs"]):
    print(f"Epoch {epoch + 1}/{CONFIG['transformer_epochs']}")
    for step, (ids, mask, labels) in enumerate(train_loader):
        optimizer.zero_grad()
        logits = hf_model(input_ids=ids, attention_mask=mask).logits
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()
        if step % 30 == 0:
            print(f"  step {step}/{len(train_loader)} loss={loss.item():.4f}")


In [ ]:
hf_model.eval()
transformer_preds = []
with torch.no_grad():
    for ids, mask in test_loader:
        logits = hf_model(input_ids=ids, attention_mask=mask).logits
        transformer_preds.extend(torch.argmax(logits, dim=1).numpy())
y_pred_transformer = np.array(transformer_preds)
print("DistilBERT evaluation complete.")


## Stage 7 — Evaluation and Comparison

Weighted precision/recall/F1 are used throughout since the classes are imbalanced — weighted averaging accounts for each class's support rather than treating all classes as equally sized.

In [ ]:
all_preds = {name: np.argmax(model.predict(X_test_pad, verbose=0), axis=1)
             for name, model in rnn_models.items()}
all_preds["DistilBERT"] = y_pred_transformer


In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

results = {}
for name, preds in all_preds.items():
    acc = accuracy_score(y_test, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, preds, average="weighted")
    results[name] = {"Accuracy": acc, "Precision": prec, "Recall": rec, "F1": f1}

comparison_df = pd.DataFrame(results).T
comparison_df.round(4)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, (name, preds) in zip(axes.flatten(), all_preds.items()):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=class_names, yticklabels=class_names)
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
plt.tight_layout()
plt.show()


In [ ]:
comparison_df.plot(kind="bar", figsize=(10, 5), colormap="viridis", ylim=(0, 1.05))
plt.title("Model Comparison")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
best_model_name = comparison_df["F1"].idxmax()
print(f"Best model: {best_model_name} (F1={comparison_df.loc[best_model_name, 'F1']:.4f})")


## Stage 8 — Save Best Model

In [ ]:
save_dir = CONFIG["save_dir"]
if os.path.exists(save_dir):
    shutil.rmtree(save_dir)
os.makedirs(save_dir, exist_ok=True)


In [ ]:
comparison_df.to_csv(os.path.join(save_dir, "comparison_results.csv"))
with open(os.path.join(save_dir, "label_encoder.pkl"), "wb") as f:
    pickle.dump(label_encoder, f)


In [ ]:
if best_model_name == "DistilBERT":
    with open(os.path.join(save_dir, "model_type.txt"), "w") as f:
        f.write("transformer")
    hf_model.save_pretrained(save_dir)
    hf_tokenizer.save_pretrained(save_dir)
else:
    with open(os.path.join(save_dir, "model_type.txt"), "w") as f:
        f.write("keras_rnn")
    rnn_models[best_model_name].save(os.path.join(save_dir, "rnn_model.keras"))
    with open(os.path.join(save_dir, "tokenizer.pkl"), "wb") as f:
        pickle.dump(tokenizer, f)
    with open(os.path.join(save_dir, "config.json"), "w") as f:
        json.dump({"max_len": MAX_LEN, "vocab_size": CONFIG["vocab_size"], "model_name": best_model_name}, f)


In [ ]:
shutil.make_archive(save_dir, "zip", save_dir)
print(f"Saved {best_model_name} to '{save_dir}/' and '{save_dir}.zip'")
